# Aprendizaje Estadístico y Data Mining

## Práctica 2: ¿Cómo son las cervezas artesanas?

### Objetivo
La misma empresa quiere encontrar una manera óptima de ordenar las cervezas para que los clientes puedan hacer unas búsquedas más eficientes en la tienda online. Para ello dispone de un set de datos de cervezas. Aplica un algoritmo de manera que se obtengan dichos grupos.

* [Link al dataset.](https://ufv-es.instructure.com/courses/42756/files/6155496/download?download_frd=1)

**Enunciado:** Realiza todo el preprocesamiento que sea necesario para adaptar las variables que no sean unívocas del dataset y poder usar el algoritmo adecuado.

**Solución**

Al empezar a trabajar con el dataset proporcionado para la práctica, nos encontramos con un problema común en muchos archivos de datos reales: el archivo CSV tiene un formato muy desordenado. Al intentar cargarlo con Python, descubrimos que muchas filas no tienen un formato uniforme. Esto se debe a que algunos campos, como las reseñas de usuarios (`review/text`) o las marcas de tiempo (`review/timeStruct`), contienen comas internas y estructuras complejas que no están correctamente delimitadas. Además, muchas filas incluyen columnas vacías innecesarias al final (como `;;;;;;;;;;;`) que contaminan la estructura esperada.

Antes de aplicar cualquier algoritmo de clustering, necesitamos que los datos estén bien estructurados. Esto implica que cada fila tenga el mismo número de columnas relevantes, que los campos de texto largos no se dividan por error, y que eliminemos cualquier ruido o información que no sea necesaria para el análisis (como datos del perfil del usuario). Si no realizamos esta limpieza previa, corremos el riesgo de que el código falle o de que los resultados del clustering sean inconsistentes o poco interpretables.

Para resolver este problema, optamos por una estrategia personalizada (a la cual nos costó llegar): leemos el archivo línea por línea como texto plano, eliminamos comillas y caracteres sobrantes, y extraemos solo las columnas relevantes para la práctica (`hasta review/timeUnix`). De forma estructurada, dividimos cada línea en tres bloques: los primeros 11 campos sencillos, el campo `review/text` que puede contener comas, y el campo `review/timeStruct`, que es un diccionario en forma de texto. Finalmente, extraemos el campo `review/timeUnix`.

Una vez extraídos los campos correctamente, validamos que la fila tenga las 14 columnas esperadas y descartamos aquellas que no cumplen. Por último, convertimos las columnas numéricas al tipo adecuado (`float o int`), transformamos `review/timeStruct` en un diccionario real para su posible análisis posterior, y almacenamos todas las filas válidas en un DataFrame limpio y consistente, listo para su uso en técnicas de clustering.

In [ ]:
import pandas as pd
import ast

In [ ]:
# Columnas necesarias para el DataFrame
# (Quitamos las relativas al usuario, por que no son necesarias
columnas = [
  'index','beer/ABV','beer/beerId','beer/brewerId','beer/name','beer/style',
  'review/appearance','review/aroma','review/overall','review/palate','review/taste',
  'review/text','review/timeStruct','review/timeUnix'
]

# Inicializar listas para almacenar filas que
# leamos del CSV y las que descartamos por error en el formato
filas = []
descartadas = []

# Abrimos el archivo CSV y lo leemos línea a línea
with open("data/cervezas.csv", "r", encoding="utf-8") as fichero:
  next(fichero) # Nos saltamos la cabecera

  # Para cada linea del fichero CSV
  for linea in fichero:
    try:
      # Eliminamos " y ; al final
      linea = linea.replace('"', '').replace(';', '').strip()

      # Paso 1: Extraemos los primeros 11 campos
      # (index, beer/ABV, beer/beerId, beer/brewerId, beer/name, beer/style,
      # review/appearance, review/aroma, review/overall, review/palate, review/taste)
      partes = linea.split(',', 11)
      if len(partes) != 12:
        descartadas.append(linea)
        continue

      primeros_11 = partes[:11]
      resto = partes[11]

      # Paso 2: Extraemos review/text y review/timeStruct
      if '{' not in resto or '}' not in resto:
        descartadas.append(linea)
        continue

      parte_texto_review, parte_struct_hora = resto.split('{', 1)
      texto_review = parte_texto_review.strip().rstrip(',')
      string_struct = '{' + parte_struct_hora.split('}', 1)[0] + '}'
      posterior_struct = parte_struct_hora.split('}', 1)[1].strip(',').split(',')

      # Paso 3: Extraemos review/timeUnix
      # (está después de review/timeStruct y es el primer campo después de la llave de cierre)
      if len(posterior_struct) == 0:
        descartadas.append(linea)
        continue

      time_unix = posterior_struct[0].strip()

      fila = primeros_11 + [texto_review, string_struct, time_unix]
      if len(fila) == len(columnas):
        filas.append(fila)
      else:
        descartadas.append(linea)

    except Exception:
      descartadas.append(linea)

# Creamos el DataFrame con las filas válidas
# y las columnas necesarias
df = pd.DataFrame(filas, columns=columnas)

# Convertimos las columnas a los tipos adecuados
columnas_numericas = [
  'beer/ABV','review/appearance','review/aroma','review/overall',
  'review/palate','review/taste','review/timeUnix'
]

# Limpiamos los espacios en blanco de las columnas de texto
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

# Convertimos las columnas numéricas a tipo float
# (si no se puede convertir, se convierte a NaN)
for col in columnas_numericas:
  df[col] = pd.to_numeric(df[col], errors='coerce')

# Convertimos review/timeStruct a diccionario
def parse_struct(val):
  try:
    return ast.literal_eval(val)
  except ValueError:
    return None

df['review/timeStruct'] = df['review/timeStruct'].apply(parse_struct)

# Resultado
print(f"Filas válidas cargadas: {len(df)}")
print(f"Filas descartadas: {len(descartadas)}")
display(df.head())


Filas válidas cargadas: 1431
Filas descartadas: 0


,index,beer/ABV,beer/beerId,beer/brewerId,beer/name,beer/style,review/appearance,review/aroma,review/overall,review/palate,review/taste,review/text,review/timeStruct,review/timeUnix
0,40163,5.0,46634,14338,Chiostro,Herbed / Spiced Beer,4.0,4.0,4.0,4.0,4.0,Pours a clouded gold with a thin white head. N...,"{'min': 38, 'hour': 3, 'mday': 16, 'sec': 10, ...",1229398690
1,8135,11.0,3003,395,Bearded Pat's Barleywine,American Barleywine,4.0,3.5,3.5,3.5,3.0,12oz bottle into 8oz snifter.\t\tDeep ruby red...,"{'min': 38, 'hour': 23, 'mday': 8, 'sec': 58, ...",1218238738
2,10529,4.7,961,365,Naughty Nellie's Ale,American Pale Ale (APA),3.5,4.0,3.5,3.5,3.5,First enjoyed at the brewpub about 2 years ago...,"{'min': 7, 'hour': 18, 'mday': 26, 'sec': 2, '...",1101492422
3,44610,4.4,429,1,Pilsner Urquell,Czech Pilsener,3.0,3.0,2.5,3.0,3.0,First thing I noticed after pouring from green...,"{'min': 7, 'hour': 1, 'mday': 20, 'sec': 5, 'y...",1308532025
4,37062,4.4,4904,1417,Black Sheep Ale (Special),English Pale Ale,4.0,3.0,3.0,3.5,2.5,A: pours an amber with a one finger head but o...,"{'min': 51, 'hour': 6, 'mday': 12, 'sec': 48, ...",1299912708


**Enunciado:** Utiliza varias configuraciones teniendo en cuenta el número de grupos que se creará y cambiando cómo se mide la distancia entre individuos. Crea una tabla donde se muestre una métrica válida para clustering y realiza un diagrama para elegir la mejor solución. ¿Cuál es?

**Solución**

**Enunciado:** Con la mejor configuración del punto anterior. Dibuja un diagrama de líneas de los centroides. Interpreta como se comporta cada grupo y selecciona cuales son las variables más influyentes.

**Solución**

**Enunciado:** Teniendo en cuenta las dos variables más influyentes e inicializando los centroides de diferente manera, dibuja cómo se van modificando los grupos y cómo van cambiando sus centroides en cada iteración. Obtén una conclusión acerca de donde deberían situarse los centroides.

**Solución**

**Enunciado:** Estudia que técnicas de postprocesamiento (variables que no aportan nada, outliers, etc) se podrían aplicar en base al error cometido en cada clúster.?

**Solución**